# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants
load_dotenv(override=True)
api_key = os.getenv('OPENROUTER_API_KEY')
if api_key and api_key.startswith('sk-or-') and len(api_key) > 10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

MODEL = 'openai/gpt-5-nano'
openai = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

API key looks good so far


In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [11]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'linkedin page', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter page', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'homepage', 'url': 'https://huggingface.co'},
  {'type': 'brand/about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Discourse community', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'product: Inference Endpoints',
   'url': 'https://endpoints.huggingface.co'},
  {'type': 'Discord', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'Docs', 'url': 'https://huggingface.co/docs'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [14]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [15]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling openai/gpt-5-nano
Found 12 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
thinkingmachines/Inkling
Updated
about 5 hours ago
•
4
•
775
prism-ml/Ternary-Bonsai-27B-gguf
Updated
2 days ago
•
74k
•
578
prism-ml/Bonsai-27B-gguf
Updated
2 days ago
•
559k
•
330
empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF
Updated
2 days ago
•
2.04M
•
2.23k
zai-org/GLM-5.2


In [26]:
#brochure_system_prompt = """
#You are an assistant that analyzes the contents of several relevant pages from a company website
#and creates a short brochure about the company for prospective customers, investors and recruits.
#Respond in markdown without code blocks.
#Include details of company culture, customers and careers/jobs if you have the information.
#"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [18]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-5-nano
Found 10 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nthinkingmachines/Inkling\nUpdated\nabout 5 hours ago\n•\n4\n•\n775\nprism-ml/Ternary-Bonsai-27B-gguf\nUpdated\n2 day

In [21]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="openai/gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [22]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-5-nano
Found 13 relevant links


# Hugging Face - The AI Community Building the Future

---

## About Hugging Face

Hugging Face is the leading collaboration platform designed exclusively for the machine learning (ML) community. It serves as a central hub where researchers, developers, and enterprises can create, discover, and collaborate on ML models, datasets, and applications.

---

## What Hugging Face Offers

- **2M+ Models**: Access and contribute to over two million machine learning models from a global community. Stay updated with trending and highly active models ranging from language processing to computer vision and beyond.

- **500k+ Datasets**: Discover and utilize extensive datasets curated for diverse ML tasks, supporting rapid experimentation and research.

- **Spaces**: Explore and deploy interactive ML applications directly through the platform, with thousands of running projects demonstrating the latest AI innovations.

- **Buckets**: Securely store and manage AI assets, facilitating scalable AI workflows.

- **Enterprise Solutions**: Tailored services including Hugging Face PRO, enterprise support, inference providers, endpoints, and storage buckets, designed to meet business needs.

- **Community and Learning**: Engage with an active and vibrant global community through discussion forums, Discord channels, daily research paper summaries, blog posts, and educational resources.

- **Open-Source Collaboration**: Hugging Face encourages open exchange through GitHub repositories where users can participate in advancing AI technology together.

---

## Company Culture

Hugging Face fosters an open and inclusive culture centered around collaboration, transparency, and innovation in AI and machine learning. The platform embodies the spirit of community-driven progress, supporting contributors from enthusiasts to industry leaders who share the vision of building the future of AI together.

---

## Customers

Hugging Face serves a diverse range of customers including:

- Independent researchers and developers looking for accessible ML resources.
- Academic institutions conducting advanced AI research.
- Startups innovating with AI-powered products.
- Large enterprises integrating AI at scale with robust support and infrastructure.
- AI practitioners seeking cutting-edge models and tools.

---

## Careers and Opportunities

Join Hugging Face to work at the forefront of AI technology in a dynamic environment that values collaboration and growth. While no specific job listings are detailed here, prospective team members can expect roles focusing on:

- Machine Learning engineering and research.
- Software development and platform engineering.
- Community engagement and support.
- Enterprise services and customer success.

Careers at Hugging Face offer the opportunity to contribute to open-source AI projects used worldwide and to help shape the future of accessible and responsible AI.

---

## Connect With Hugging Face

Discover more and get involved:

- Visit the [Hugging Face Website](https://huggingface.co)
- Join the community on [Discord](https://discord.gg/huggingface)
- Explore open-source projects on [GitHub](https://github.com/huggingface)
- Participate in forums and discussions
- Access learning resources and daily AI research updates

---

Embrace the future of machine learning with Hugging Face — where the AI community builds tomorrow, today.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [23]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="openai/gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [24]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-5-nano
Found 8 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is a vibrant AI community dedicated to building the future of machine learning. As a leading platform, it enables collaboration among developers, researchers, and enterprises on models, datasets, and AI applications. The platform hosts over 2 million machine learning models and half a million datasets, making it the home of machine learning innovation and discovery.

---

## What We Offer

- **Extensive Model Library**: Access and contribute to over 2 million machine learning models covering a wide range of AI tasks.
- **Large Dataset Repository**: Explore more than 500,000 datasets for training and evaluation.
- **Spaces**: A hub for running and sharing AI applications, including cutting-edge image and video generation tools.
- **Collaboration Platform**: Host, discover, and collaborate on unlimited public models, datasets, and applications.
- **Enterprise Solutions**: Tailored support and tools for businesses including Hugging Face PRO, enterprise support, inference providers, endpoints, and storage solutions.

---

## Community and Culture

Hugging Face fosters a collaborative and inclusive AI community. It encourages open-source contributions and knowledge sharing, reflected in active forums, Discord channels, blogs, and daily research paper discussions. The emphasis on community involvement drives rapid innovation and democratizes AI technology for everyone.

---

## Our Customers

From individual AI researchers and hobbyists to large enterprises, Hugging Face serves a diverse customer base. Our platform supports:

- Data scientists and machine learning engineers seeking reliable and up-to-date AI models.
- Developers building applications with state-of-the-art AI capabilities.
- Enterprises requiring scalable and secure AI solutions with comprehensive support.
- Researchers focused on advancing AI science and sharing their work globally.

---

## Careers at Hugging Face

Join a passionate team shaping the future of AI. Hugging Face offers exciting opportunities for talents in machine learning research, engineering, product management, community engagement, and enterprise support. The company values innovation, collaboration, and continuous learning to empower employees to make meaningful contributions to AI advancements.

Visit our website for current job openings and learn how you can be part of the AI revolution.

---

## Connect with Hugging Face

Explore AI apps, models, and datasets at [huggingface.co](https://huggingface.co)  
Join the community on Discord, GitHub, and forums to collaborate and learn  
Follow our blog for the latest AI insights and updates

---

**Hugging Face**  
*The AI community building the future.*

In [27]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling openai/gpt-5-nano
Found 12 relevant links


# Welcome to Hugging Face: The AI Community Building the Future!

## Who Are We?
Imagine a place where AI enthusiasts, researchers, coders, and dreamers gather around a digital campfire to build, share, and revolutionize AI. That’s Hugging Face. We are the *home of machine learning*, a collaboration platform where the world’s best minds and machines come together to create, discover, and collaborate on cutting-edge machine learning models, datasets, and applications.

## What We Offer
- **2 Million+ Models**: From the quirky to the complex, explore a huge library of models updated daily — ready to make your AI projects smarter and cooler.
- **500K+ Datasets**: Fuel your AI creativity with massive datasets, constantly growing and updated to keep you on the cutting edge.
- **Spaces & Applications**: Try out and create AI-powered apps right in your browser — from turning images into videos to editing like a pro.
- **Open Collaboration**: Unlimited hosting and sharing in the AI community. Work openly with peers or go enterprise-level with premium services.

## Who Uses Hugging Face?
- AI researchers hunting for the latest breakthroughs.
- Developers craving easy access to pre-trained models.
- Enterprises looking to integrate AI with professional support.
- Hobbyists and students eager to learn and create.
- Creative professionals pushing the boundaries of image and video generation.

## Our Culture: Collaboration with a Hug 🤗
We’re not just about code and datasets; we’re about people who love AI and believe in open collaboration. Our community spirit is alive in:
- A buzzing **Discord and Forum** where ideas, jokes, and breakthroughs flow freely.
- A cozy corner on **GitHub** where open-source projects bloom daily.
- A welcoming environment for learners at all levels—whether you want to reproduce ICML papers or just tinker with your first model.

## Careers: Join the Machine Intelligence Party
Want to work where the future of AI is coded daily? At Hugging Face, you'll:
- Collaborate with passionate pros and inspiring newcomers alike.
- Have your work impact a vibrant global AI community.
- Push the boundaries with projects that mix hardcore science and infinite creativity.
- Enjoy a culture that’s as friendly as it is innovative—where your ideas (and memes) matter.

## The Hugging Face Experience: More Than Just a Tech Company
- **HuggingChat**: Chat with AI that’s as savvy and sassy as your smartest friend.
- **Enterprise Solutions**: For businesses that want to power their AI dreams without the headaches.
- **Learn, Share, Repeat**: Daily papers, blogs, and events to keep your brain buzzing.

---

### Ready to Hug the Future? 
Dive in at [huggingface.co](https://huggingface.co)  
Because building the future is better when you do it together.

---

***Hugging Face — Making AI Collaborative, Open, and a Little More Fun.***

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>